## 1. Define the Schema of the Table

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
from pyspark.sql import SparkSession

# Initialize Spark session (already available in Databricks)
spark = SparkSession.builder.getOrCreate()

# Define the schema for the financial table
schema_financial = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("account", StringType(), nullable=False),
    StructField("transaction_type", StringType(), nullable=False),
    StructField("amount", DoubleType(), nullable=False),
    StructField("transaction_date", TimestampType(), nullable=False)
])


## 2. Create 10 Fake Records

In [ ]:
from pyspark.sql.functions import current_timestamp

# Create a list of 10 fake records
fake_data = [
    (1, "Account_A", "Deposit", 1000.0, current_timestamp()),
    (2, "Account_B", "Withdrawal", 200.0, current_timestamp()),
    (3, "Account_C", "Transfer", 500.0, current_timestamp()),
    (4, "Account_D", "Payment", 300.0, current_timestamp()),
    (5, "Account_E", "Deposit", 1500.0, current_timestamp()),
    (6, "Account_F", "Withdrawal", 100.0, current_timestamp()),
    (7, "Account_G", "Transfer", 750.0, current_timestamp()),
    (8, "Account_H", "Payment", 400.0, current_timestamp()),
    (9, "Account_I", "Deposit", 2000.0, current_timestamp()),
    (10, "Account_J", "Withdrawal", 50.0, current_timestamp())
]

# Create DataFrame with fake records
df_initial = spark.createDataFrame(fake_data, schema=schema_financial)


## 3. Write the Data into a Delta Table

In [ ]:
# Define the path for the Delta table
delta_table_path = "/mnt/datalake/financial/financial_table_delta"

# Write the initial DataFrame as a Delta table
df_initial.write.format("delta").mode("overwrite").save(delta_table_path)


## 4. Simulate Operations: UPDATE, DELETE, INSERT, and MERGE
First, load the Delta table:

In [ ]:
from delta.tables import DeltaTable

# Load the existing Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)


### Simulate an UPDATE:

In [ ]:
# For example, increase the amount of transactions in Account_A by 10%
delta_table.update(
    condition = "account = 'Account_A'",
    set = { "amount": "amount * 1.10" }
)


### Simulate a DELETE:

In [ ]:
# Delete records where amount is less than 100
delta_table.delete("amount < 100")


### Simulate an INSERT:

In [ ]:
# Create new records to insert
new_data = [
    (11, "Account_K", "Deposit", 600.0, current_timestamp()),
    (12, "Account_L", "Withdrawal", 300.0, current_timestamp())
]
df_new = spark.createDataFrame(new_data, schema=schema_financial)

# Insert new records into the Delta table
df_new.write.format("delta").mode("append").save(delta_table_path)


### Simulate a MERGE:

In [ ]:
# Create a DataFrame with records to merge
merge_data = [
    (3, "Account_C", "Transfer", 550.0, current_timestamp()),  # Update existing value
    (13, "Account_M", "Payment", 800.0, current_timestamp())   # New record
]
df_merge = spark.createDataFrame(merge_data, schema=schema_financial)

# Execute merge: update if id exists, insert if not
delta_table.alias("tgt").merge(
    source = df_merge.alias("src"),
    condition = "tgt.id = src.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


## 5. Extract Data from the History
After performing the operations, extract a new DataFrame with audit information:

> Note: Some columns like userName, userMetadata, operationParameters.engine may not be present or may be null depending on your environment and how operations were executed. Adjust accordingly based on available metadata.

In [ ]:
# Retrieve the complete history of the table
history_df = delta_table.history()

# Select and rename columns for auditing
audit_history = history_df.selectExpr(
    "operation as Operation",
    "timestamp as Date",
    "userName as User",               # May vary depending on how user info is captured
    "userMetadata as Job",            # Additional info if available
    "operationParameters.engine as Engine",  # Example column for engine, adjust as needed
    "operationMetrics.numUpdatedRows as UpdatedRows",
    "operationMetrics.numInsertedRows as InsertedRows",
    "operationMetrics.numDeletedRows as DeletedRows"
)

audit_history.show(truncate=False)



## 6. Create an Audit Table for Future Updates

Create or update a separate Delta table to store the extracted audit logs, which will be fed as new operations occur:

In [ ]:
# Define the path for the audit table
audit_table_path = "/mnt/datalake/financial/audit_table"

# Write the audit DataFrame to a Delta table (overwrite mode for initialization)
audit_history.write.format("delta").mode("overwrite").save(audit_table_path)

# Register as a SQL table if desired
spark.sql(f"CREATE TABLE IF NOT EXISTS financial_audit USING DELTA LOCATION '{audit_table_path}'")


## 7. Update the Audit Table with New Operations (Example)
To continuously feed the audit table with new operations, you can schedule a job or use triggers to:

- Extract recent operations from the main table.

- Insert or update records in the audit table.

A basic periodic update example:

In [ ]:
# Get the last audited version
last_audited_version = history_df.agg({"version": "max"}).collect()[0][0]

# Extract new operations after the last audited version
new_operations_df = delta_table.history().filter(f"version > {last_audited_version}")

# Process and select the desired fields
new_audit_operations = new_operations_df.selectExpr(
    "operation as Operation",
    "timestamp as Date",
    "userName as User",
    "userMetadata as Job",
    "operationParameters.engine as Engine",
    "operationMetrics.numUpdatedRows as UpdatedRows",
    "operationMetrics.numInsertedRows as InsertedRows",
    "operationMetrics.numDeletedRows as DeletedRows"
)

# Append new operations to the audit table
new_audit_operations.write.format("delta").mode("append").save(audit_table_path)
